# CANguard -- PIRD Evaluation (thin orchestrator)

Fully drives the library: window features, residual transform, Isolation Forest,
evaluation orchestration, and visualization. No algorithm code is duplicated here.


## 1. Load + features


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from canguard.data import get_loader
from canguard.detectors import IsolationForestDetector
from canguard.evaluation import cross_attack_evaluate, train_anomaly_detector
from canguard.features import (
    BEHAVIORAL_FEATURES_V1 as FEATURES,
)
from canguard.features import (
    FeaturePipeline,
    fit_known_ids_on_normal_prefix,
    fit_per_id_stats,
    temporal_split,
    transform_residuals,
)
from canguard.visualization import (
    plot_cross_attack_matrix,
    plot_roc_pr,
    plot_score_distribution_grid,
    plot_score_timeline,
)

plt.rcParams['figure.dpi'] = 120
DATA_DIR = Path('HCRL Car-Hacking')
SAMPLE_SIZE = 60000
NAMES = ['DoS', 'Fuzzy', 'RPM', 'gear']


## 2. Build feature tables + residual splits


In [ ]:
samples = {
    name: get_loader('hcrl', DATA_DIR / f'{name}_dataset.csv').load(sample_size=SAMPLE_SIZE)
    for name in NAMES
}
WINDOW_SIZE = 30
feature_tables = {}
for name, df in samples.items():
    known = fit_known_ids_on_normal_prefix(df)
    pipe = FeaturePipeline(window_size=WINDOW_SIZE, known_ids=known)
    feature_tables[name] = pipe.process_dataframe(df)
    pipe.reset()

RES_COLS = [c + '_res' for c in FEATURES]
residual_data = {}
raw_data = {}
for name in NAMES:
    ft = feature_tables[name]
    calib, train, test = temporal_split(ft, 0.4, 0.2, 0.4)
    raw_data[name] = {'calib': calib, 'train': train, 'test': test}
    stats, global_stats = fit_per_id_stats(calib, FEATURES)
    residual_data[name] = {
        'calib': transform_residuals(calib, stats, global_stats, FEATURES),
        'train': transform_residuals(train, stats, global_stats, FEATURES),
        'test': transform_residuals(test, stats, global_stats, FEATURES),
        'stats': stats, 'global_stats': global_stats,
    }
    print(f'{name}: calib/train/test residuals ready')


## 3. Residual Isolation Forest (primary)


In [ ]:
results = {}
for name in NAMES:
    rd = residual_data[name]
    det = IsolationForestDetector(n_estimators=200, random_state=0)
    results[name] = train_anomaly_detector(det, rd['train'], rd['test'], RES_COLS)
    r = results[name]
    print(f'{name:>5s}: F1={r["f1"]:.3f}  Recall={r["recall"]:.3f}  FPR={r["fpr"]:.4f}  ROC-AUC={r["roc_auc"]:.3f}')


## 4. Diagnostic plots (stored scores, no recomputation)


In [ ]:
plot_score_distribution_grid(results, NAMES)
plt.show()


In [ ]:
r = results['RPM']
plot_roc_pr(r['y_test'], r['scores_test'], fpr_at_op=r['fpr'], title_prefix='RPM ')
plt.show()


In [ ]:
rd = residual_data['RPM']
# timeline needs chronological scores; reuse the stored test scores from results
plot_score_timeline(results['RPM']['scores_test'], results['RPM']['y_test'],
                    results['RPM']['threshold'], title='RPM test segment')
plt.show()


## 5. Cross-attack matrix (single residualization, target stats)


In [ ]:
import pandas as pd

target_stats = {
    t: fit_per_id_stats(raw_data[t]['calib'], FEATURES)
    for t in NAMES
}
def cross_attack(src, tgt):
    stats, gstats = target_stats[tgt]
    src_norm = raw_data[src]['train']
    src_norm = src_norm[src_norm['is_attack'] == 0]
    src_res = transform_residuals(src_norm, stats, gstats, FEATURES)
    tgt_res = transform_residuals(raw_data[tgt]['test'], stats, gstats, FEATURES)
    det = IsolationForestDetector(n_estimators=200, random_state=0)
    return cross_attack_evaluate(det, src_res, tgt_res, RES_COLS)
rows = []
for s in NAMES:
    for t in NAMES:
        r = cross_attack(s, t)
        rows.append({'src': s, 'tgt': t, 'recall': r['recall'], 'fpr': r['fpr']})
cross_df = pd.DataFrame(rows)
rec_mat = cross_df.pivot_table(index='src', columns='tgt', values='recall', aggfunc='first')
fpr_mat = cross_df.pivot_table(index='src', columns='tgt', values='fpr', aggfunc='first')
print('Recall matrix:'); print(rec_mat.to_string())
plot_cross_attack_matrix(rec_mat, fpr_mat)
plt.show()
